### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="horse_colic_survival",
    dataset_year="1989",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C58W23",
    download_description="""
Get the train and test data from UCI.

wget https://archive.ics.uci.edu/static/public/47/horse+colic.zip && unzip horse+colic.zip horse-colic.data horse-colic.test && rm horse+colic.zip && mkdir -p local-data-warehouse/horse_colic_survival && mv horse-colic.data horse-colic.test local-data-warehouse/horse_colic_survival/
""",
    # References
    academic_reference_bibtex="""@misc{McLeish1989HorseColic,
  author       = {McLeish, Mary and Cecile, Matt},
  title        = {{Horse Colic}},
  year         = {1989},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C58W23}
}
""",
    academic_reference_bibtex_key="McLeish1989HorseColic",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI and merge the train and test data.

- We encode missing values as NaN.
- We drop features that leak the target.
- We keep only the first instance (hospital number) per horse, as they can be treated multiple times and otherwise leaks information about the target.
- We use the target "outcome" to model a task where we aim to predict if the horse survives or not. We drop cases that have not outcome recorded.
- We keep the duplicates occurring in the data.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="outcome",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="outcome",
)

## Preprocessing

In [2]:
import pandas as pd

columns = [
    "surgery","age","hospital_number","rectal_temperature","pulse","respiratory_rate","temperature_of_extremities","peripheral_pulse","mucous_membranes","capillary_refill_time","pain","peristalsis","abdominal_distension","nasogastric_tube","nasogastric_reflux","nasogastric_reflux_ph","rectal_exam_feces","abdomen","packed_cell_volume","total_protein","abdominocentesis_appearance","abdominocentesis_total_protein","outcome","surgical_lesion","lesion_type_1","lesion_type_2","lesion_type_3","cp_data"
]

df = pd.read_csv(dataset_mold.path / "horse-colic.data", header=None, names=columns, na_values="?", sep="\s+")
df_test = pd.read_csv(dataset_mold.path / "horse-colic.test", header=None, names=columns, na_values="?", sep="\s+")
df = pd.concat([df, df_test], ignore_index=True)
print("Loaded data shape:", df.shape)

# Drop duplicates by hospital number, keeping the first instance, as they can be treated multiple times and otherwise leaks information about the target.
df = df.drop_duplicates(subset=["hospital_number"], keep="first")

# Only keep non nan outcome rows
df = df[~df["outcome"].isna()]

df = df.drop(columns=[
    "hospital_number", # constant now
    "surgery", # leaks target
    "cp_data", # no significance according to original data description
    # Other target outcomes and thus leaking of outcome
    "surgical_lesion", "lesion_type_1", "lesion_type_2", "lesion_type_3",
])
df["outcome"] = df["outcome"].map({1: "Lived", 2: "Died", 3: "Euthanized"})
as_cat_type = [
    "temperature_of_extremities", "peripheral_pulse", "mucous_membranes", "capillary_refill_time", "pain", "peristalsis", "abdominal_distension", "nasogastric_tube",
    "nasogastric_reflux", "rectal_exam_feces", "abdomen", "abdominocentesis_appearance", "outcome",
]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (368, 28)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 344
Columns: 21
Use sampling: False (sample size: 344)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['total_protein', 'packed_cell_volume', 'pulse', 'abdominocentesis_total_protein', 'rectal_temperature', 'respiratory_rate', 'nasogastric_reflux_ph', 'mucous_membranes', 'pain', 'abdomen']
Rows remaining as candidates after top-10 filter: 4 (of 344)

#### Duplicate Report
Total duplicate rows: 2 (0.58% of dataset)
Duplicate rows ignoring target: 3 (0.87% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,rectal_temperature,pulse,respiratory_rate,temperature_of_extremities,peripheral_pulse,mucous_membranes,capillary_refill_time,pain,peristalsis,abdominal_distension,nasogastric_tube,nasogastric_reflux,nasogastric_reflux_ph,rectal_exam_feces,abdomen,packed_cell_volume,total_protein,abdominocentesis_appearance,abdominocentesis_total_protein,outcome
0,1,37.5,48.0,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,5.0,41.0,55.0,3.0,2.0,Euthanized
1,1,38.0,66.0,20.0,1.0,3.0,3.0,1.0,5.0,3.0,1.0,1.0,1.0,NaN,3.0,NaN,46.0,46.0,3.0,2.0,Euthanized
2,1,36.1,88.0,NaN,3.0,3.0,3.0,1.0,3.0,3.0,2.0,2.0,3.0,NaN,NaN,4.0,45.0,7.0,3.0,4.8,Euthanized
3,1,38.0,76.0,18.0,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71.0,11.0,NaN,NaN,Lived
4,1,37.5,44.0,NaN,1.0,1.0,1.0,1.0,3.0,3.0,2.0,NaN,NaN,NaN,NaN,NaN,45.0,5.8,2.0,1.4,Lived


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,abdominocentesis_appearance,category,182.0,52.91,3.0,"2.0, 3.0, 1.0"
1,abdomen,category,137.0,39.83,5.0,"5.0, 4.0, 1.0, 2.0, 3.0"
2,nasogastric_reflux,category,127.0,36.92,3.0,"1.0, 3.0, 2.0"
3,nasogastric_tube,category,126.0,36.63,3.0,"2.0, 1.0, 3.0"
4,rectal_exam_feces,category,123.0,35.76,4.0,"4.0, 1.0, 3.0, 2.0"
5,peripheral_pulse,category,82.0,23.84,4.0,"1.0, 3.0, 4.0, 2.0"
6,temperature_of_extremities,category,64.0,18.60,4.0,"3.0, 1.0, 2.0, 4.0"
7,abdominal_distension,category,64.0,18.60,4.0,"1.0, 3.0, 2.0, 4.0"
8,pain,category,62.0,18.02,5.0,"3.0, 2.0, 5.0, 1.0, 4.0"
9,peristalsis,category,50.0,14.53,4.0,"3.0, 4.0, 1.0, 2.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,344.0,1.604651,2.117698,1.0,9.0
rectal_temperature,279.0,38.127240,0.719895,35.4,40.8
pulse,320.0,70.812500,28.499725,30.0,184.0
respiratory_rate,277.0,30.682310,18.050256,8.0,96.0
nasogastric_reflux_ph,63.0,4.860317,1.969898,1.0,8.5
packed_cell_volume,309.0,45.457605,10.994502,4.0,75.0
total_protein,304.0,25.366118,27.906170,3.3,89.0
abdominocentesis_total_protein,124.0,2.870161,1.875429,0.1,10.1


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                      rank                          
abdomen                     1           <NA>    137  39.83
                            2            5.0     88  25.58
                            3            4.0     53  15.41
                            4            1.0     25   7.27
                            5            2.0     23   6.69
abdominal_distension        1            1.0     93  27.03
                            2            3.0     79  22.97
                            3            2.0     70  20.35
                            4           <NA>     64  18.60
                            5            4.0     38  11.05
abdominocentesis_appearance 1           <NA>    182  52.91
                            2            2.0     58  16.86
                            3            3.0     56  16.28
                            4            1.0     48  13.95
capillary_refill_time       1            1.0    220  63.95
                            2            2.0     85  24.71
                            3           <NA>     37  10.76
                            4            3.0      2   0.58
mucous_membranes            1            1.0     92  26.74
                            2            3.0     72  20.93
                            3            4.0     48  13.95
                            4           <NA>     48  13.95
                            5            2.0     37  10.76
nasogastric_reflux          1            1.0    130  37.79
                            2           <NA>    127  36.92
                            3            3.0     45  13.08
                            4            2.0     42  12.21
nasogastric_tube            1           <NA>    126  36.63
                            2            2.0    111  32.27
                            3            1.0     83  24.13
                            4            3.0     24   6.98
outcome                     1          Lived    215  62.50
                            2           Died     79  22.97
                            3     Euthanized     50  14.53
pain                        1            3.0     77  22.38
                            2            2.0     72  20.93
                            3           <NA>     62  18.02
                            4            5.0     46  13.37
                            5            1.0     44  12.79
peripheral_pulse            1            1.0    139  40.41
                            2            3.0    108  31.40
                            3           <NA>     82  23.84
                            4            4.0     10   2.91
                            5            2.0      5   1.45
peristalsis                 1            3.0    139  40.41
                            2            4.0     89  25.87
                            3           <NA>     50  14.53
                            4            1.0     46  13.37
                            5            2.0     20   5.81
rectal_exam_feces           1           <NA>    123  35.76
                            2            4.0     91  26.45
                            3            1.0     61  17.73
                            4            3.0     56  16.28
                            5            2.0     13   3.78
temperature_of_extremities  1            3.0    123  35.76
                            2            1.0     87  25.29
                            3           <NA>     64  18.60
                            4            2.0     39  11.34
                            5            4.0     31   9.01

In [8]:
# Target Distribution
target_df

,count,pct
outcome,,
Lived,215,62.50
Died,79,22.97
Euthanized,50,14.53


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to horse_colic_survival/019d736d-db15-7ad1-b407-b937aa958bd9


019d736d-db15-7ad1-b407-b937aa958bd9
c1af125fbd852f123bf84946aebfe7c8450353c5460ea7d8185610b6a0eefa98
